# Star Cluster Injection Demo

Injection into Rubin calexp images following the official DP0.2 tutorial pattern.
Reference: [DP02_14_Injecting_Synthetic_Sources](https://github.com/rubin-dp0/tutorial-notebooks/blob/main/DP02_14_Injecting_Synthetic_Sources.ipynb)

## 0. Setup

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.patches import Circle
from scipy.signal import fftconvolve

import astropy.units as u
from astropy.coordinates import SkyCoord

USERNAME = os.environ.get('USER', 'your_username')
PIPELINE_PATH = f'/home/{USERNAME}/INJECT/star-cluster-injection-pipeline'
if os.path.exists(PIPELINE_PATH):
    sys.path.insert(0, PIPELINE_PATH)
    print(f'Pipeline: {PIPELINE_PATH}')
else:
    print(f'Pipeline not found at {PIPELINE_PATH}')

from src.light_profiles import PlummerProfile, KingProfile, EFFProfile, SersicProfile, mag_to_flux
print('Imports OK')

## 1. Load Calexp Image (exactly as in DP0.2 tutorial)

In [ ]:
# Fix working directory
os.chdir(os.path.dirname(os.path.abspath("__file__")))

# Exact Butler pattern from the official DP0.2 tutorial
from lsst.daf.butler import Butler
from lsst.geom import Point2D
import lsst.geom as geom

butler = Butler('dp02', collections='2.2i/runs/DP0.2')
print('Butler connected')

In [ ]:
# Find calexp images overlapping a tract — same approach as the tutorial
tract = 4431
where = f"instrument='LSSTCam-imSim' AND skymap='DC2' AND tract={tract} AND detector=19 AND band='i'"
calexp_refs = sorted(list(set(butler.query_datasets('calexp', where=where))))
print(f'Found {len(calexp_refs)} calexp datasets')

if len(calexp_refs) == 0:
    # Try a different tract
    where = "instrument='LSSTCam-imSim' AND skymap='DC2' AND tract=3828 AND detector=19 AND band='i'"
    calexp_refs = sorted(list(set(butler.query_datasets('calexp', where=where))))
    print(f'  Trying tract 3828: found {len(calexp_refs)}')

In [ ]:
# Pick one calexp — same as tutorial (index 5)
dataId = calexp_refs[5].dataId
print(f'dataId = {dataId}')

calexp = butler.get('calexp', dataId=dataId)

# Get WCS, bounding box, sky coords — exactly as in tutorial
wcs   = calexp.getWcs()
bbox  = calexp.getBBox()
psf   = calexp.getPsf()
photo_calib = calexp.getPhotoCalib()

boxcen = bbox.getCenter()
cen    = wcs.pixelToSky(boxcen)
sc_cen = SkyCoord(ra=cen[0].asDegrees()*u.deg, dec=cen[1].asDegrees()*u.deg)

image  = calexp.image.array.copy()
ny, nx = image.shape

print(f'Calexp shape   : {image.shape}')
print(f'BBox           : {bbox}')
print(f'Centre (ra,dec): {sc_cen}')
print(f'PSF type       : {type(psf).__name__}')
print(f'Image range    : [{image.min():.1f}, {image.max():.1f}]')

ZEROPOINT = 31.4   # Rubin AB nJy
PIXEL_SCALE = wcs.getPixelScale().asArcseconds()
print(f'Pixel scale    : {PIXEL_SCALE:.4f} arcsec/px')

In [ ]:
# Quick look at the calexp
fig, ax = plt.subplots(figsize=(8, 8))
v1, v2 = np.percentile(image, [1, 99])
ax.imshow(image, cmap='gray', origin='lower', vmin=v1, vmax=v2)
ax.set_title(f'Calexp — {dataId}')
ax.set_xlabel('X (pixels)'); ax.set_ylabel('Y (pixels)')
plt.tight_layout(); plt.show()

## 2. Extract PSF from Calexp

In [ ]:
# Scan for a valid PSF position (calexp PSF can also fail at edges)
print('Scanning PSF validity...')
margin = 200
valid_positions, invalid_positions = [], []

for y in np.linspace(margin, ny - margin, 8):
    for x in np.linspace(margin, nx - margin, 8):
        pos = Point2D(float(x), float(y))
        try:
            psf.computeImage(pos)
            sh = psf.computeShape(pos)
            fw = 2.355 * np.sqrt((sh.getIxx() + sh.getIyy()) / 2.0)
            valid_positions.append({'x': float(x), 'y': float(y), 'fwhm': fw})
        except Exception as e:
            invalid_positions.append({'x': float(x), 'y': float(y), 'error': str(e)})

print(f'  Valid: {len(valid_positions)}, Invalid: {len(invalid_positions)}')

if not valid_positions:
    raise RuntimeError('No valid PSF positions found!')

# Reference PSF
ref = valid_positions[len(valid_positions)//2]
ref_pt = Point2D(ref['x'], ref['y'])
psf_arr = psf.computeImage(ref_pt).array.copy()
psf_arr = psf_arr / psf_arr.sum()
psf_fwhm = ref['fwhm']

print(f'\nPSF at ({ref["x"]:.0f}, {ref["y"]:.0f}):')
print(f'  FWHM   : {psf_fwhm:.3f} px = {psf_fwhm * PIXEL_SCALE:.3f} arcsec')
print(f'  Kernel : {psf_arr.shape}')
print(f'  Sum    : {psf_arr.sum():.6f}')
fwhm_vals = [v['fwhm'] for v in valid_positions]
print(f'  FWHM range: [{min(fwhm_vals):.3f}, {max(fwhm_vals):.3f}] px')

In [ ]:
# PSF visualization
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(psf_arr, cmap='inferno', origin='lower')
axes[0].set_title(f'PSF kernel {psf_arr.shape}')

psf_clip = np.clip(psf_arr, 1e-10, None)
axes[1].imshow(psf_clip, cmap='inferno', origin='lower',
               norm=LogNorm(vmin=psf_clip[psf_clip > 0].min(), vmax=psf_clip.max()))
axes[1].set_title('Log scale (wings)')

cy_k, cx_k = psf_arr.shape[0]//2, psf_arr.shape[1]//2
yg, xg = np.mgrid[:psf_arr.shape[0], :psf_arr.shape[1]]
rr = np.sqrt((xg-cx_k)**2 + (yg-cy_k)**2)
r_bins = np.arange(0, min(cx_k, cy_k), 0.5)
r_cen  = 0.5*(r_bins[:-1] + r_bins[1:])
rprof  = [psf_arr[(rr >= r_bins[i]) & (rr < r_bins[i+1])].mean() for i in range(len(r_bins)-1)]
axes[2].semilogy(r_cen, rprof, 'k-', lw=2)
axes[2].axvline(psf_fwhm/2, color='red', ls='--', label=f'HWHM={psf_fwhm/2:.2f} px')
axes[2].set_xlabel('Radius (px)'); axes[2].set_ylabel('Mean PSF value')
axes[2].set_title(f'Radial profile — FWHM={psf_fwhm:.2f} px ({psf_fwhm*PIXEL_SCALE:.2f}")')
axes[2].legend(); axes[2].grid(alpha=0.3)

fig.suptitle(f'PSF from calexp — {dataId}', fontsize=12)
plt.tight_layout()
plt.savefig('fig_psf_calexp.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_psf_calexp.png')

## 3. Plot: 2D Light Profiles

In [ ]:
flux = mag_to_flux(21.0, zero_point=ZEROPOINT)
r_half = 10.0
stamp_size = 101
half = stamp_size // 2
yy, xx = np.mgrid[:stamp_size, :stamp_size]
dx, dy = xx - half, yy - half

profiles = {
    'Plummer': PlummerProfile(total_flux=flux, r_half=r_half),
    'King':    KingProfile(total_flux=flux, r_half=r_half, concentration=30),
    'EFF':     EFFProfile(total_flux=flux, r_half=r_half, gamma=2.5),
    'Sersic':  SersicProfile(total_flux=flux, r_half=r_half, sersic_n=2.0),
}

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for i, (name, prof) in enumerate(profiles.items()):
    stamp = prof.surface_brightness(dx, dy)
    stamp_pos = np.clip(stamp, 1e-12, None)

    im = axes[0, i].imshow(stamp_pos, cmap='inferno', origin='lower',
                           norm=LogNorm(vmin=stamp_pos.max()*1e-4, vmax=stamp_pos.max()))
    axes[0, i].add_patch(Circle((half, half), r_half, fill=False, ec='cyan', lw=1.5, ls='--'))
    axes[0, i].set_title(f'{name}\nr_half={r_half} px', fontsize=11)
    if i == 0: axes[0, i].set_ylabel('Y (pixels)')
    plt.colorbar(im, ax=axes[0, i], shrink=0.8)

    rr = np.sqrt(dx**2 + dy**2).ravel()
    vals = stamp.ravel()
    order = np.argsort(rr)
    axes[1, i].semilogy(rr[order], vals[order], '.', ms=0.5, alpha=0.3, color='navy')
    axes[1, i].axvline(r_half, color='red', ls='--', lw=1.5, label=f'r_half={r_half}')
    axes[1, i].set_xlabel('Radius (px)')
    if i == 0: axes[1, i].set_ylabel('Surface brightness')
    axes[1, i].set_title('Radial profile')
    axes[1, i].legend(fontsize=8); axes[1, i].grid(alpha=0.3); axes[1, i].set_xlim(0, half)

fig.suptitle(f'2D Light Profiles (mag=21, r_half={r_half} px)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('fig_light_profiles.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_light_profiles.png')

## 4. Plot: Raw vs PSF-Convolved Profiles

In [ ]:
test_mags   = [20.0, 22.0, 24.0]
test_rhalfs = [5.0, 10.0, 20.0]

fig, axes = plt.subplots(3, 3, figsize=(14, 14))
for row, mag in enumerate(test_mags):
    for col, rh in enumerate(test_rhalfs):
        f = mag_to_flux(mag, zero_point=ZEROPOINT)
        prof = PlummerProfile(total_flux=f, r_half=rh)
        sz = max(61, int(rh * 10) | 1)
        h = sz // 2
        yg, xg = np.mgrid[:sz, :sz]
        raw  = prof.surface_brightness(xg - h, yg - h)
        conv = fftconvolve(raw, psf_arr, mode='same')

        combined = np.zeros_like(raw)
        combined[:, :h] = conv[:, :h]
        combined[:, h:] = raw[:, h:]
        axes[row, col].imshow(combined, cmap='inferno', origin='lower',
                              vmin=0, vmax=max(raw.max(), conv.max()) * 0.3)
        axes[row, col].axvline(h, color='white', lw=0.8, alpha=0.7)
        axes[row, col].text(h*0.3, sz-5, 'PSF-conv', color='white', fontsize=8, ha='center')
        axes[row, col].text(h*1.7, sz-5, 'Raw',      color='white', fontsize=8, ha='center')
        axes[row, col].set_title(f'mag={mag}, r_half={rh} px', fontsize=10)
        if col == 0: axes[row, col].set_ylabel(f'mag={mag}')
        axes[row, col].tick_params(labelsize=6)

fig.suptitle(f'Raw vs PSF-convolved Plummer (FWHM={psf_fwhm:.1f} px = {psf_fwhm*PIXEL_SCALE:.2f}")', fontsize=13)
plt.tight_layout()
plt.savefig('fig_psf_convolution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_psf_convolution.png')

## 5. Inject Clusters into the Calexp

In [ ]:
np.random.seed(42)

N_CLUSTERS  = 30
MAG_MIN, MAG_MAX       = 19.0, 25.0
R_HALF_MIN, R_HALF_MAX = 3.0, 25.0

# Use only valid PSF region
vxs = np.array([v['x'] for v in valid_positions])
vys = np.array([v['y'] for v in valid_positions])
xlo, xhi = int(vxs.min()), int(vxs.max())
ylo, yhi = int(vys.min()), int(vys.max())
print(f'Valid PSF region: x=[{xlo},{xhi}], y=[{ylo},{yhi}]')

injected_image = image.copy().astype(np.float64)
injection_info = []
n_skipped = 0
attempts  = 0

while len(injection_info) < N_CLUSTERS and attempts < N_CLUSTERS * 20:
    attempts += 1
    x   = np.random.randint(xlo, xhi)
    y   = np.random.randint(ylo, yhi)
    mag = np.random.uniform(MAG_MIN, MAG_MAX)
    rh  = 10 ** np.random.uniform(np.log10(R_HALF_MIN), np.log10(R_HALF_MAX))
    fl  = mag_to_flux(mag, zero_point=ZEROPOINT)

    ptype = np.random.choice(['plummer', 'king', 'eff', 'sersic'])
    if   ptype == 'plummer': prof = PlummerProfile(total_flux=fl, r_half=rh)
    elif ptype == 'king':    prof = KingProfile(total_flux=fl, r_half=rh, concentration=30)
    elif ptype == 'eff':     prof = EFFProfile(total_flux=fl, r_half=rh, gamma=2.5)
    else:                    prof = SersicProfile(total_flux=fl, r_half=rh, sersic_n=2.0)

    sz = max(61, int(rh * 10) | 1)
    h  = sz // 2
    yg, xg = np.mgrid[:sz, :sz]
    raw_stamp = prof.surface_brightness(xg - h, yg - h)

    # Get local PSF at this position — skip if invalid
    pos = Point2D(float(x), float(y))
    try:
        local_psf = psf.computeImage(pos).array.copy()
        local_psf = local_psf / local_psf.sum()
    except:
        n_skipped += 1
        continue

    conv_stamp = fftconvolve(raw_stamp, local_psf, mode='same')
    pm = conv_stamp > 0
    noisy = conv_stamp.copy()
    noisy[pm] = np.random.poisson(conv_stamp[pm])

    y0, y1 = max(0, y-h), min(ny, y-h+sz)
    x0, x1 = max(0, x-h), min(nx, x-h+sz)
    sy0 = y0-(y-h); sy1 = sy0+(y1-y0)
    sx0 = x0-(x-h); sx1 = sx0+(x1-x0)
    injected_image[y0:y1, x0:x1] += noisy[sy0:sy1, sx0:sx1]

    injection_info.append({
        'id': len(injection_info), 'x': x, 'y': y,
        'magnitude': mag, 'r_half': rh, 'profile': ptype,
        'flux_injected': float(noisy.sum()),
        'peak': float(noisy.max()),
        'raw_stamp': raw_stamp, 'conv_stamp': conv_stamp,
    })

injected_image = injected_image.astype(image.dtype)
print(f'Injected {len(injection_info)} clusters ({n_skipped} skipped — invalid PSF)')

## 6. Plot: Before / After / Difference

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
v1, v2 = np.percentile(image, [1, 99])

axes[0].imshow(image, cmap='gray', origin='lower', vmin=v1, vmax=v2)
axes[0].set_title('Original Calexp', fontsize=12)

axes[1].imshow(injected_image, cmap='gray', origin='lower', vmin=v1, vmax=v2)
for info in injection_info:
    c = plt.cm.plasma(np.clip((info['magnitude']-MAG_MIN)/(MAG_MAX-MAG_MIN), 0, 1))
    axes[1].add_patch(Circle((info['x'], info['y']), info['r_half']*2, fill=False, ec=c, lw=1.2, alpha=0.8))
axes[1].set_title(f'After Injection ({len(injection_info)} clusters)', fontsize=12)

diff = injected_image.astype(float) - image.astype(float)
diff[diff <= 0] = np.nan
valid_diff = diff[np.isfinite(diff)]
if len(valid_diff) > 0:
    im = axes[2].imshow(diff, cmap='hot', origin='lower',
                        norm=LogNorm(vmin=np.percentile(valid_diff, 5), vmax=np.nanmax(valid_diff)))
    plt.colorbar(im, ax=axes[2], shrink=0.8, label='Injected flux')
axes[2].set_title('Difference (injected signal only)', fontsize=12)

for ax in axes:
    ax.set_xlabel('X (pixels)'); ax.set_ylabel('Y (pixels)')

fig.suptitle(f'Star Cluster Injection — calexp  {dataId}', fontsize=13)
plt.tight_layout()
plt.savefig('fig_before_after_diff.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_before_after_diff.png')

## 7. Plot: Postage Stamps (Before / After / Cluster Only)

In [ ]:
sorted_info = sorted(injection_info, key=lambda d: d['magnitude'])
n_stamps = min(8, len(sorted_info))
picked = [sorted_info[i] for i in np.linspace(0, len(sorted_info)-1, n_stamps, dtype=int)]

fig, axes = plt.subplots(3, n_stamps, figsize=(2.5*n_stamps, 7.5))
for col, info in enumerate(picked):
    cx_i, cy_i = int(info['x']), int(info['y'])
    r = max(int(info['r_half']*4), 25)
    y0, y1 = max(0, cy_i-r), min(ny, cy_i+r)
    x0, x1 = max(0, cx_i-r), min(nx, cx_i+r)

    s_orig = image[y0:y1, x0:x1].astype(float)
    s_inj  = injected_image[y0:y1, x0:x1].astype(float)
    s_diff = s_inj - s_orig
    vp1, vp2 = np.percentile(s_inj, [1, 99.5])

    axes[0, col].imshow(s_orig, cmap='gray', origin='lower', vmin=vp1, vmax=vp2)
    axes[0, col].set_title(f'mag={info["magnitude"]:.1f}', fontsize=9)
    axes[1, col].imshow(s_inj, cmap='gray', origin='lower', vmin=vp1, vmax=vp2)
    axes[1, col].set_title(f'r_half={info["r_half"]:.1f}px', fontsize=9)
    d_pos = np.clip(s_diff, 0, None)
    if d_pos.max() > 0:
        axes[2, col].imshow(d_pos, cmap='inferno', origin='lower',
                            norm=LogNorm(vmin=max(d_pos[d_pos>0].min(), 1e-2), vmax=d_pos.max()))
    axes[2, col].set_title(info['profile'], fontsize=9)
    for row in range(3):
        axes[row, col].plot(cx_i-x0, cy_i-y0, 'r+', ms=8, mew=1)
        axes[row, col].tick_params(labelsize=5)

axes[0, 0].set_ylabel('Before', fontsize=11)
axes[1, 0].set_ylabel('After', fontsize=11)
axes[2, 0].set_ylabel('Cluster only', fontsize=11)
fig.suptitle('Postage Stamps — bright to faint', fontsize=13)
plt.tight_layout()
plt.savefig('fig_postage_stamps.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_postage_stamps.png')

## 8. Plot: Zoomed Bright / Median / Faint

In [ ]:
all_mags = [i['magnitude'] for i in injection_info]
med_mag  = np.median(all_mags)
bright = min(injection_info, key=lambda d: d['magnitude'])
mid    = min(injection_info, key=lambda d: abs(d['magnitude'] - med_mag))
faint  = max(injection_info, key=lambda d: d['magnitude'])

fig, axes = plt.subplots(3, 3, figsize=(14, 14))
for row, (info, label) in enumerate(zip([bright, mid, faint], ['Brightest', 'Median mag', 'Faintest'])):
    cx_i, cy_i = int(info['x']), int(info['y'])
    r = max(int(info['r_half']*5), 40)
    y0, y1 = max(0, cy_i-r), min(ny, cy_i+r)
    x0, x1 = max(0, cx_i-r), min(nx, cx_i+r)
    s_orig = image[y0:y1, x0:x1].astype(float)
    s_inj  = injected_image[y0:y1, x0:x1].astype(float)
    s_diff = s_inj - s_orig
    vp1, vp2 = np.percentile(s_inj, [0.5, 99.5])

    axes[row, 0].imshow(s_orig, cmap='gray', origin='lower', vmin=vp1, vmax=vp2)
    axes[row, 0].set_ylabel(f'{label}\nmag={info["magnitude"]:.1f}\nr_half={info["r_half"]:.1f}px\n{info["profile"]}', fontsize=10)
    axes[row, 1].imshow(s_inj, cmap='gray', origin='lower', vmin=vp1, vmax=vp2)
    axes[row, 1].add_patch(Circle((cx_i-x0, cy_i-y0), info['r_half'], fill=False, ec='cyan', lw=1.5, ls='--'))
    d = np.clip(s_diff, 0, None)
    if d.max() > 0:
        axes[row, 2].imshow(d, cmap='inferno', origin='lower',
                            norm=LogNorm(vmin=max(1e-2, d[d>0].min()), vmax=d.max()))
    for col in range(3):
        axes[row, col].plot(cx_i-x0, cy_i-y0, 'r+', ms=12, mew=1.5)

axes[0, 0].set_title('Before', fontsize=12)
axes[0, 1].set_title('After', fontsize=12)
axes[0, 2].set_title('Cluster Only', fontsize=12)
fig.suptitle('Zoomed Examples — Bright / Median / Faint', fontsize=14)
plt.tight_layout()
plt.savefig('fig_zoomed_examples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_zoomed_examples.png')

## 9. Plot: Summary Diagnostic Panel

In [ ]:
from collections import Counter

mags   = [i['magnitude'] for i in injection_info]
sizes  = [i['r_half'] for i in injection_info]
fluxes = [i['flux_injected'] for i in injection_info]
peaks  = [i['peak'] for i in injection_info]
bg_std = np.std(image)
snrs   = [p / bg_std for p in peaks]
counts = Counter([i['profile'] for i in injection_info])

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

axes[0,0].hist(mags, bins=12, edgecolor='black', color='steelblue', alpha=0.8)
axes[0,0].axvline(np.median(mags), color='red', ls='--', lw=1.5, label=f'median={np.median(mags):.1f}')
axes[0,0].set_xlabel('Magnitude'); axes[0,0].set_ylabel('Count')
axes[0,0].set_title('Magnitude Distribution'); axes[0,0].legend()

axes[0,1].hist(sizes, bins=12, edgecolor='black', color='darkorange', alpha=0.8)
axes[0,1].axvline(np.median(sizes), color='red', ls='--', lw=1.5, label=f'median={np.median(sizes):.1f}')
axes[0,1].set_xlabel('r_half (pixels)'); axes[0,1].set_ylabel('Count')
axes[0,1].set_title('Size Distribution'); axes[0,1].legend()

axes[0,2].pie(counts.values(), labels=counts.keys(), autopct='%1.0f%%',
              colors=plt.cm.Set2.colors[:len(counts)])
axes[0,2].set_title('Profile Types')

sc = axes[1,0].scatter(mags, np.log10(np.array(fluxes)+1), c=sizes, cmap='plasma', s=40, alpha=0.8)
plt.colorbar(sc, ax=axes[1,0], label='r_half (px)')
axes[1,0].set_xlabel('Magnitude'); axes[1,0].set_ylabel('log10(Flux)')
axes[1,0].set_title('Flux vs Magnitude')

axes[1,1].scatter(mags, snrs, c=sizes, cmap='plasma', s=40, alpha=0.8)
axes[1,1].axhline(3, color='red', ls='--', label='S/N=3')
axes[1,1].axhline(5, color='orange', ls='--', label='S/N=5')
axes[1,1].set_xlabel('Magnitude'); axes[1,1].set_ylabel('Peak S/N')
axes[1,1].set_title('Peak S/N vs Magnitude')
axes[1,1].set_yscale('log'); axes[1,1].legend(fontsize=8)

axes[1,2].axis('off')
summary = (
    f'Injection Summary\n'
    f'N clusters  : {len(injection_info)}\n'
    f'Image type  : calexp\n'
    f'Band        : i\n'
    f'Image shape : {image.shape}\n'
    f'Pixel scale : {PIXEL_SCALE:.4f} arcsec/px\n'
    f'Mag range   : [{min(mags):.1f}, {max(mags):.1f}]\n'
    f'r_half range: [{min(sizes):.1f}, {max(sizes):.1f}] px\n'
    f'PSF FWHM    : {psf_fwhm:.2f} px ({psf_fwhm*PIXEL_SCALE:.2f}")\n'
    f'PSF kernel  : {psf_arr.shape}\n'
    f'BG std      : {bg_std:.2f}\n'
    f'Median S/N  : {np.median(snrs):.1f}\n'
    f'S/N > 3     : {sum(1 for s in snrs if s>3)}/{len(injection_info)}\n'
    f'S/N > 5     : {sum(1 for s in snrs if s>5)}/{len(injection_info)}\n'
    f'Profiles    : {dict(counts)}'
)
axes[1,2].text(0.05, 0.95, summary, transform=axes[1,2].transAxes,
               fontsize=9, va='top', family='monospace',
               bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

fig.suptitle('Injection Diagnostic Summary', fontsize=14)
plt.tight_layout()
plt.savefig('fig_summary_panel.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_summary_panel.png')

In [ ]:
import glob
figs = sorted(glob.glob('fig_*.png'))
print(f'Generated {len(figs)} figures:')
for f in figs:
    print(f'  {f:40s} {os.path.getsize(f)/1024:6.0f} KB')